# Project 92 — Local Test Case Generator
## Analyze Code → Generate Comprehensive Tests

**Stack:** LangChain · Ollama · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Source Code to Test

In [ ]:
source_code = '''
class ShoppingCart:
    def __init__(self):
        self.items = {}
        self.discount_code = None

    def add_item(self, name, price, quantity=1):
        if price < 0:
            raise ValueError("Price cannot be negative")
        if quantity < 1:
            raise ValueError("Quantity must be at least 1")
        if name in self.items:
            self.items[name]["quantity"] += quantity
        else:
            self.items[name] = {"price": price, "quantity": quantity}

    def remove_item(self, name):
        if name not in self.items:
            raise KeyError(f"Item {name} not in cart")
        del self.items[name]

    def get_total(self):
        total = sum(item["price"] * item["quantity"] for item in self.items.values())
        if self.discount_code == "SAVE10":
            total *= 0.9
        return round(total, 2)

    def apply_discount(self, code):
        valid_codes = ["SAVE10", "HALF50"]
        if code not in valid_codes:
            raise ValueError(f"Invalid discount code: {code}")
        self.discount_code = code
'''
print(source_code)

## Step 2 — Generate Test Suite

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.2)

test_prompt = ChatPromptTemplate.from_messages([
    ("system", """Generate a comprehensive pytest test suite for the given code.
Include:
- Happy path tests
- Edge cases (empty cart, zero values, etc.)
- Error cases (invalid input, missing items)
- Boundary tests
- Integration tests (multiple operations)

Use descriptive test names. Include at least 10 test functions.
Return ONLY valid Python code."""),
    ("human", "Generate tests for:\n{code}")
])
test_chain = test_prompt | llm | StrOutputParser()

tests = test_chain.invoke({"code": source_code})
print("GENERATED TEST SUITE")
print("="*50)
print(tests)

## Step 3 — Validate Generated Tests

In [ ]:
# Execute the generated tests in-memory
import io, contextlib

# First, execute the source code
exec(source_code)

# Try to run the test code
output = io.StringIO()
try:
    with contextlib.redirect_stdout(output):
        exec(tests)
    print("✓ Tests parsed and loaded successfully")
    print(output.getvalue())
except SyntaxError as e:
    print(f"Syntax error in generated tests: {e}")
except Exception as e:
    print(f"Runtime note: {e}")
    print("(Some tests may need pytest runner to execute fully)")

# Count test functions
test_count = tests.count("def test_")
print(f"\nGenerated {test_count} test functions")

## What You Learned
- **Automated test generation** from source code
- **Comprehensive test coverage** including edge/error cases
- **Test validation** by parsing generated code